In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from rich.console import Console  # only for fancy text formatting
from anthropic import Anthropic

load_dotenv(override=True)
console = Console(force_jupyter=False)

client = Anthropic()
MODEL = "claude-haiku-4-5"
MAX_TOKENS = 1024
DATASET_FILE_NAME = "eval_dataset.json"

In [2]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": MODEL,
        "max_tokens": MAX_TOKENS,
        "messages": messages,
        "stop_sequences": stop_sequences,
        # this will work with older (<1.1.0) SDK
        # "temperature": temperature,
        # ------------------------------
        # for 1.1.0+ SDK use the following
        "extra_body": {"temperature": temperature},
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [3]:
import json


def generate_dataset():
    prompt = """
        Generate a evaluation dataset for a prompt evaluation. The dataset will be used
        to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related 
        tasks. Generate an array of JSON objects, each representing task that requires Python, 
        JSON, or a Regex to complete. 

        Example output:
        ```json
        [
            {
                "task": "Description of task",
                "format": "json" or "python" or "regex"
            },
            ...additional
        ]
        ```

        * Focus on tasks that can be solved by writing a single Python function, a single 
          JSON object, or a regular expression.
        * Focus on tasks that do not require writing much code

        Please generate 3 objects.
    """

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    response = chat(messages, stop_sequences=["```"])
    return json.loads(response)

In [4]:
# let's test the function defined above

dataset = generate_dataset()
console.print(dataset)

# write the dataset to JSON file
with open(DATASET_FILE_NAME, "w") as f:
    json.dump(dataset, f, indent=2)

[
    {
        'task': 'Extract all S3 bucket names from AWS CloudFormation template 
output',
        'format': 'regex'
    },
    {
        'task': 'Parse an IAM policy document and return a JSON object with 
only the Actions and Resources fields',
        'format': 'json'
    },
    {
        'task': 'Write a Python function that takes an AWS region code and 
returns the corresponding region name',
        'format': 'python'
    }
]


### Running the Evals

The `run_prompt` function below is not defining any formatting instructions, so expect a lot of text to be returned from Claude.

In [5]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the
    result"""

    # NOTE: test_case is one of JSON element from the sample
    # JSON above
    prompt = f"""
        Please solve the following task:

        {test_case["task"]}
    """

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [6]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    # TODO - Grading
    score = 10

    return {
        "test_case": test_case,
        "output": output,
        "score": score,
    }

In [7]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    return results

In [8]:
# open the test-cases JSON and run the evaluations
import json

with open(DATASET_FILE_NAME, "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)
console.print(results)

[
    {
        'test_case': {
            'task': 'Extract all S3 bucket names from AWS CloudFormation 
template output',
            'format': 'regex'
        },
        'output': "# Extract S3 Bucket Names from CloudFormation 
Template\n\nI'd be happy to help extract S3 bucket names from an AWS 
CloudFormation template. However, I don't see a template provided in your 
message.\n\nPlease share your CloudFormation template (in JSON or YAML format),
and I'll extract all S3 bucket names for you.\n\n## What I can extract:\n\n1. 
**Explicit bucket names** - defined in `BucketName` properties\n2. **Logical 
IDs** - CloudFormation resource logical identifiers for S3 buckets\n3. 
**Resource references** - buckets created via `AWS::S3::Bucket` resources\n\n##
Example:\n\nIf you provide a template like:\n```yaml\nResources:\n  MyBucket:\n
Type: AWS::S3::Bucket\n    Properties:\n      BucketName: my-app-bucket-prod\n 
\n  LoggingBucket:\n    Type: AWS::S3::Bucket\n    Properties:\n      
Bucke

## Model Based Grading

Model graders feed your original output into another API call for evaluation. This approach offers tremendous flexibility for assessing:

* Response quality
* Quality of instruction following
* Completeness
* Helpfulness
* Safety

Here's how to build a model grader - this will replace our hard-coded `10` value in the above listed code.

In [9]:
def grade_by_model(test_case, output):
    # Create evaluation prompt
    eval_prompt = f"""
    You are an expert code reviewer. Your task is to evaluate the following AI-generated solution.

    Original Task:
    <task>
    {test_case["task"]}
    </task>

    Solution to Evaluate:
    <solution>
    {output}
    </solution>

    Output Format:    
    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    
    Respond with JSON. Keep your response concise and direct.
    Example response shape:
    {{
        "strengths": string[],
        "weaknesses": string[],
        "reasoning": string,
        "score": number
    }}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")

    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

Now let's re-implement the evaluation code above. We show all functions again below:

In [10]:
def run_prompt2(test_case):
    """Merges the prompt and test case input, then returns the
    result"""

    # NOTE: test_case is one of JSON element from the sample
    # JSON above
    prompt = f"""
        Please solve the following task:

        {test_case["task"]}
    """

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [11]:
def run_test_case2(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt2(test_case)

    # here we replace the hard-coded score with model
    # assisted scoring
    model_grade = grade_by_model(test_case, output)

    score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    return {
        "test_case": test_case,
        "output": output,
        "score": score,
        "reasoning": reasoning,
    }

In [12]:
from statistics import mean


def run_eval2(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case2(test_case)
        # you will get original test case, model response
        # model score and model reasoning in result
        results.append(result)

    average_score = mean([result["score"] for result in results])
    # print(f"Average score: {average_score}")

    return results, average_score

In [13]:
# open the test-cases JSON and run the evaluations
import json

with open(DATASET_FILE_NAME, "r") as f:
    dataset = json.load(f)

results, average_score = run_eval2(dataset)
console.print(results)
console.print(f"[red]Average score:[/red] {average_score}")

[
    {
        'test_case': {
            'task': 'Extract all S3 bucket names from AWS CloudFormation 
template output',
            'format': 'regex'
        },
        'output': '# S3 Bucket Name Extraction from CloudFormation 
Template\n\nI\'d be happy to help extract S3 bucket names from an AWS 
CloudFormation template! However, I don\'t see a template in your 
message.\n\nPlease provide the CloudFormation template (in JSON or YAML 
format), and I\'ll extract all S3 bucket names for you.\n\n## Expected Input 
Format\n\nYou can share your template as:\n- **JSON**: 
`{"AWSTemplateFormatVersion": "2010-09-09", ...}`\n- 
**YAML**:\n```yaml\nAWSTemplateFormatVersion: \'2010-09-09\'\nResources:\n  
MyBucket:\n    Type: AWS::S3::Bucket\n    ...\n```\n\n## What I\'ll 
Extract\n\nOnce you provide the template, I\'ll identify:\n- ✅ Bucket names 
defined in `Properties.BucketName`\n- ✅ Bucket references in other 
resources\n- ✅ Bucket names passed via parameters\n- ✅ Any other S3 bucket 
id

## Code Graders

Next up, we need to implement our `Code Grader`. Our code grader will take in some output from the model and make sure that we get back just plain Python, or plain JSON, or a RegEx without any kind of explanation. In addition, we should also make sure that we get valid syntax what whatever type of code we actually got. We use a little trick for this. We'll define 3 helper functions - `validate_json(...)`, `validate_python(...)` and `validate_regex(...)`. Each of these functions will take the output from the model and either try to parse it as JSON, or a Python Abstract Syntax Tree (AST) or compile it as a regular expression. If parsing is successful, we'll return a score of 10, else we return 0.

In order to know which validator/grading must be called, we'll need our test-case dataset to include expected format. We'll update the prompt that generates our dataset to do so. 

* **Step 1:** Add functions to validate JSON/Regex/Python
* **Step 2:** Ensure our dataset contains the type of output expected from model
* **Step 3:** Update draft prompt to make it clear that we want only JSON/Python/Regex
* **Step 4:** Add functions to validate JSON/Regex/Python
* **Step 5:** Merge scores from model grader & code grader

First let's add the validation functions for JSON, Python and Regex. These will respectively try to `loads` JSON, `parse` Python using AST and `compile` a Regex. If successful, it returns a score of `10`, otherwise it returns `0`. For Python, we are using the **Abstract Syntax Tree (`ast`)** package, which is part of the standard library.


In [14]:
import re
import ast


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

Now let's add the modified functions to evaluate the dataset using both model evaluation & code evaluation.

In [15]:
def run_prompt3(test_case):
    """Merges the prompt and test case input, then returns the
    result"""

    # NOTE: test_case is one of JSON element from the sample
    # JSON above
    prompt = f"""
        Please solve the following task:

        {test_case["task"]}

        * Respond only with Python, JSON, or a plain Regex
        * Do not add any comments or commentary or explanation
    """

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output


def run_test_case3(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt3(test_case)

    # here we replace the hard-coded score with model
    # assisted scoring
    model_grade = grade_by_model(test_case, output)

    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2

    return {
        "test_case": test_case,
        "output": output,
        "score": score,
        "reasoning": reasoning,
    }


from statistics import mean


def run_eval3(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case3(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    # print(f"Average score: {average_score}")

    return results, average_score

In [16]:
with open(DATASET_FILE_NAME, "r") as f:
    dataset = json.load(f)

results, average_score = run_eval3(dataset)
console.print(f"[red]Evaluation score:[/red] {average_score:.4f}")
console.print(results)

Evaluation score: 4.1667
[
    {
        'test_case': {
            'task': 'Extract all S3 bucket names from AWS CloudFormation 
template output',
            'format': 'regex'
        },
        'output': '\n```python\nimport json\nimport re\n\ndef 
extract_s3_buckets(template_content):\n    """Extract all S3 bucket names from 
AWS CloudFormation template output"""\n    \n    # Try to parse as JSON first\n
try:\n        template = json.loads(template_content)\n    except 
(json.JSONDecodeError, TypeError):\n        template = template_content\n    \n
s3_buckets = set()\n    \n    # Convert to string if it\'s a dict for regex 
searching\n    template_str = json.dumps(template) if isinstance(template, 
dict) else str(template)\n    \n    # Pattern 1: Direct bucket name 
references\n    pattern1 = r\'"BucketName"\\s*:\\s*"([a-z0-9.-]+)"\'\n    
matches = re.findall(pattern1, template_str)\n    s3_buckets.update(matches)\n 
\n    # Pattern 2: S3 URI patterns (s3://bucket-name)\n    patte